# 第 38 课：标点恢复、ITN 与文本规范化

ASR token 往往是“spoken form”。用户需要“written form”：数字、日期、金额、单位、大小写和标点。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 后处理与语义 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 37 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 标点恢复、ITN、文本规范化 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：标点恢复、ITN、文本规范化。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：raw ASR 与 normalized text 的证据边界；置信度；N-best；会话状态隔离。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](37_时间戳_说话人分段与Diarization.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：带时间、说话人、候选与置信度的可追溯识别结果
  ↓ 本课要学会的变换、状态或判断
输出：保留原证据、可校准、可拒绝或澄清的文本/语义结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

import re

项目根目录: <REPO_ROOT>


## 1. ITN 不是普通字符串替换

“一二三”可能是号码 123，也可能是逐字读数；“一百零二”是 102。上下文、locale 和业务领域会改变规则。

In [2]:
digit_map=dict(zip("零一二三四五六七八九","0123456789"))
def digit_sequence_itn(text):
    return re.sub(r"[零一二三四五六七八九]{2,}",lambda m:"".join(digit_map[c] for c in m.group()),text)
for s in ["电话一三八零零一二三四五六","编号一二三","一百零二元"]:print(s,"->",digit_sequence_itn(s))

电话一三八零零一二三四五六 -> 电话13800123456
编号一二三 -> 编号123
一百零二元 -> 一百02元


最后一个例子故意展示失败：简单逐字映射不理解“百”。成熟 ITN 常用分类器 + 规则/FST，把不同 semiotic class（数字、日期、货币等）分别处理。

## 2. 标点恢复会影响语义

In [3]:
examples=[("如果下雨就不去",["如果下雨，就不去。","如果下雨就不去。"]),("他说你不行",["他说：你不行。","他说你不行。"])]
for raw,cands in examples:print("raw",raw,"candidates",cands)

raw 如果下雨就不去 candidates ['如果下雨，就不去。', '如果下雨就不去。']
raw 他说你不行 candidates ['他说：你不行。', '他说你不行。']


流式标点通常会修订最近若干词，因此也需要 partial/stable 机制。不能让标点模块无限回改已经提交的业务文本。

## 3. Normalization 与 ITN 方向相反

- Text normalization：`102元` → “一百零二元”，常用于 TTS/训练文本。
- Inverse text normalization：口语识别结果 → `102元`。

训练标签规范必须和 tokenizer、解码评估、线上展示一致。

## 本课测试

1. “一二三”和“一百二十三”能否用同一逐字规则？
2. 标点是否可能改变意图？
3. 流式标点为什么需要可修订区域？
4. WER 评估前为什么要统一 normalization？
5. ITN 出错时应优先保留原文还是编造规范形式？

<details><summary>展开参考答案</summary>

1. 不能。2. 可以。3. 后续词会改变句法判断。4. 否则格式差异被误算成识别错误。5. 应可追溯地保留 spoken form，避免生成错误事实。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 38 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `标点恢复`、`ITN`、`文本规范化`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**逐字数字规则把“一百零二”变错**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**为号码、数量、日期设计分类规则和反例**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把 spoken/verbatim/normalized 三层分开**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：标点恢复、ITN、文本规范化。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 标点恢复、ITN、文本规范化。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
